In [1]:
import pickle

In [19]:
%matplotlib inline

In [21]:
#importing necessary libraries
import matplotlib.pyplot as plt
import numpy as np
from mplsoccer import Pitch, Sbopen, VerticalPitch
import pandas as pd
from scipy.stats.contingency import odds_ratio
from warnings import filterwarnings
filterwarnings('ignore')

# Import EURO2024 matches

In [5]:
with open('list_euro2024_matches.pkl', 'rb') as f:
    list_euro2024_matches = pickle.load(f) # deserialize using load()

# Import Teams Names

In [8]:
with open('teams_names.pkl', 'rb') as f:
    teams_names = pickle.load(f) # deserialize using load()

# Import Teams Matches

In [13]:
with open('df_teams_matches.pkl', 'rb') as f:
    df_teams_matches = pickle.load(f) # deserialize using load()

In [26]:
parser = Sbopen()

In [50]:
def uppor_player_cumulative(team_name,player):
    odds = []
    passes_long_pressure=0
    passes_long_not_pressure=0
    passes_not_long_pressure=0
    passes_not_long_not_pressure=0
    for match in df_teams_matches[df_teams_matches.Team==team_name].Matches.to_list()[0]:
        #parser = Sbopen()
        df= parser.event(match)[0]
        passes = df.loc[df['team_name'] == "Portugal"].loc[df['type_name'] == 'Pass'].loc[df['sub_type_name'] != 'Throw-in'].loc[df['sub_type_name'] != 'Corner'].loc[df['sub_type_name'] != 'Free Kick'].loc[df['sub_type_name'] != 'Kick Off'].loc[df['sub_type_name'] != 'Goal Kick'].set_index('id')  
        df_pass = passes.loc[passes.player_name==player, ['x', 'y', 'end_x', 'end_y','under_pressure','pass_length']]
        df_pass_long=df_pass.loc[df_pass.pass_length>10.9361]
        df_pass_not_long=df_pass.loc[df_pass.pass_length<=10.9361]
        df_pass_pressure=df_pass.loc[df_pass.under_pressure==1]
        df_pass_not_pressure=df_pass.loc[df_pass.under_pressure!=1]
        df_pass_long_pressure=df_pass_long.loc[df_pass_long.under_pressure==1]
        df_pass_long_not_pressure=df_pass_long.loc[df_pass_long.under_pressure!=1]
        df_pass_not_long_pressure=df_pass_not_long.loc[df_pass_not_long.under_pressure==1]
        df_pass_not_long_not_pressure=df_pass_not_long.loc[df_pass_not_long.under_pressure!=1]
        passes_long_pressure += len(df_pass_long_pressure)
        passes_long_not_pressure += len(df_pass_long_not_pressure)
        passes_not_long_pressure += len(df_pass_not_long_pressure)
        passes_not_long_not_pressure += len(df_pass_not_long_not_pressure)
        res = odds_ratio([[passes_long_pressure,passes_long_not_pressure],[passes_not_long_pressure,passes_not_long_not_pressure]],kind='sample')
        number = [[passes_long_pressure,passes_long_not_pressure],[passes_not_long_pressure,passes_not_long_not_pressure]]
    odds.append((player,number,res.statistic,res.confidence_interval(confidence_level=0.95)))
    df_odds=pd.DataFrame(odds,columns=["Nome","Passes","UPPOR","Intervalo de confiança"])
    return df_odds

In [52]:
def list_players(team_name):
    list_team_players = []
    for match in df_teams_matches[df_teams_matches.Team==team_name].Matches.to_list()[0]: 
        df= parser.event(match)[0]
        passes = df.loc[df['team_name'] == team_name].loc[df['type_name'] == 'Pass'].loc[df['sub_type_name'] != 'Throw-in'].loc[df['sub_type_name'] != 'Corner'].loc[df['sub_type_name'] != 'Free Kick'].loc[df['sub_type_name'] != 'Kick Off'].loc[df['sub_type_name'] != 'Goal Kick'].set_index('id')
        for name in passes.player_name.unique():
            if name not in list_team_players: 
                list_team_players.append(name)
    return list_team_players

In [62]:
def uppor_all_players_cumulative(team_name):
    df_uppor=pd.DataFrame()
    for name in list_players(team_name):
        df_uppor=pd.concat([df_uppor,uppor_player_cumulative(team_name,name)])
    return df_uppor.sort_values(by="UPPOR")

In [64]:
uppor_all_players_cumulative(team_name="Portugal")

,Nome,Passes,UPPOR,Intervalo de confiança
0,Nélson Cabral Semedo,"[[0, 28], [3, 19]]",0.000000,"(0.0, nan)"
0,Danilo Luís Hélio Pereira,"[[0, 76], [2, 8]]",0.000000,"(0.0, nan)"
0,Matheus Luiz Nunes,"[[2, 10], [2, 3]]",0.300000,"(0.02871088598790419, 3.134699501712233)"
0,António João Pereira Albuquerque Tavares Silva,"[[3, 48], [2, 12]]",0.375000,"(0.05621586756976507, 2.501517917970072)"
0,Nuno Mendes,"[[20, 224], [14, 68]]",0.433673,"(0.20796587659965127, 0.9043439295229432)"
0,Gonçalo Bernardo Inácio,"[[6, 97], [5, 40]]",0.494845,"(0.14282174529661934, 1.714528348755346)"
0,João Maria Lobo Alves Palhinha Gonçalves,"[[15, 115], [11, 42]]",0.498024,"(0.21190483130236865, 1.1704670421688876)"
0,Rafael Alexandre Conceição Leão,"[[11, 70], [6, 25]]",0.654762,"(0.21914605616426458, 1.9562896062619015)"
0,Vitor Machado Ferreira,"[[32, 198], [15, 64]]",0.689562,"(0.3511101665653283, 1.3542648332796572)"
0,João Pedro Cavaco Cancelo,"[[19, 152], [9, 50]]",0.694444,"(0.29532039778891406, 1.632982652164964)"


In [68]:
df_uppor_all=pd.DataFrame()
for team in teams_names:
    df_uppor_all=pd.concat([df_uppor_all,uppor_all_players_cumulative(team_name=team)])  

In [69]:
df_uppor_all

,Nome,Passes,UPPOR,Intervalo de confiança
0,Virgil van Dijk,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Nathan Aké,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Stefan de Vrij,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Jerdy Schouten,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Denzel Dumfries,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
...,...,...,...,...
0,Krzysztof Piątek,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Robert Lewandowski,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Kamil Grosicki,"[[0, 0], [0, 0]]",NaN,"(0, inf)"
0,Łukasz Skorupski,"[[0, 0], [0, 0]]",NaN,"(0, inf)"


In [ ]:
df_uppor_all.sort_values(by=["UPPOR"], ascending=False)